https://arxiv.org/pdf/2005.11401

<h1> Parsing </h1>

<p> 
Consiste em transformar documentos em texto estruturado. <br>
Tem que preservar a estrutura e o contexto do documento para garantir que o mesmo se mantém legível. <br>
É uma excelente oportunidade para adicionar metadados. 
</p>

https://www.thinkevolveconsulting.com/rag-engineers-guide-to-document-parsing/ <br>
https://pypi.org/project/ragparser/ <br>
https://unstructured.io/blog/chunking-for-rag-best-practices

In [ ]:
from unstructured.partition.text import partition_text #https://docs.unstructured.io/open-source/core-functionality/partitioning
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.docx import partition_docx

from pathlib import Path
from langchain_core.documents import Document #https://reference.langchain.com/python/langchain-core/documents

In [23]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs") #path para pastas de documentos

documentos = []

for docs in path.iterdir():
    #print (docs.suffix)
    file_dtype = docs.suffix.lower()
    print (file_dtype)

    if file_dtype == ".txt":
        parsing = partition_text(str(docs))

    elif file_dtype == ".pdf":
        parsing = partition_pdf(str(docs))

    elif file_dtype == ".docx":
        parsing = partition_docx(str(docs))

    text = "\n".join([str(el) for el in parsing])

    documentos.append (
        Document (
            page_content = text,
            metadata = {
                "title": docs.name
            }
        )
    )


.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
No languages specified, defaulting to English.


.txt
.txt
.txt
.docx


In [24]:
documentos

[Document(metadata={'title': 'Go Forward and Transform.pdf'}, page_content='Go Forward and Transform Arquitetura Transformer e a Evolução dos Modelos de Linguagem Modernos\nEliezer Carvalho eliezer.carvalho.11.8a@gmail.com\nAbstracto\nOs Transformers [1] constituem uma arquitetura de Redes Neurais Artificiais fundamentada em mecanismos de Attention. Num período recente, estes modelos estabeleceram-se como a base dominante no Processamento de Linguagem Natural, impulsionando o desenvolvimento de Large Language Models (LLMs) com capacidades avançadas de geração e compreensão textual. O presente estudo propõe uma exploração e compreensão da arquitetura dos Transformers. É apresentada uma visão abrangente do ecossistema contemporâneo de LLMs, incluindo técnicas e ferramentas fundamentais neste domínio. Este projeto apresenta-se como um guia teórico para a introdução ao Processamento de Linguagem Natural (NLP), permitindo a aquisição de uma compreensão sólida das tecnologias subjacentes aos

In [ ]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs") #path para pastas de documentos

documentos = [] #lista
id = 0 #idx

for docs in path.iterdir(): #iterar sobre os documentos na pasta
    elementos = partition (str(docs)) #aplicar partition a cada documento

    for x in elementos:
        documentos.append ({
            "id": f"{docs.name}_{id}",
            "page_content": x.text,
            "metadata": {
                "source": docs.name,
                "type": x.category,
                "filetype": x.metadata.to_dict()["filetype"]
            }
        })
        id += 1

'''
elements = partition (path)

for el in elements:
    print (el.text)
    print (el.category)
    print (el.metadata.to_dict())
'''

#display (documentos)
#display (documentos [0]["page_content"])
#display (documentos[2]["metadata"])

In [ ]:
#Como vamos trabalhar com LangChain é necessário converter o parsing para uma classe Document

documentos = [ #Sobreposição de variável para evitar confusão. Esta variável documentos é a variável que sai da fase do Parsing
    Document (
        page_content = x["page_content"], 
        metadata = {
            "id": x["id"],
            "source": x["metadata"]["source"],
            "type": x["metadata"]["type"],
            "file_type": x["metadata"]["filetype"]
        }
    )
    for x in documentos
]

#display (documentos)
#display (documentos[0])
#display (documentos[1].metadata)
#display (documentos[2].page_content)

---------------------------------------------------------------------------

<h1> Chunking </h1>

<p>
Chunking é uma técnica que consiste em converter documentos maiores em unidades mais pequenas (chunks). <br>
Large Language Models têm um problema complicado com context window e por esse motivo não é inteligente e eficiente despejar documentos inteiros nos modelos. <br>
O Chunking acaba por se tornar uma etapa importante na implementação de um sistema RAG porque um Chunk inteligente e eficaz faz toda a diferença.
</p>

https://community.databricks.com/t5/technical-blog/the-ultimate-guide-to-chunking-strategies-for-rag-applications/ba-p/113089

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #https://reference.langchain.com/python/langchain-text-splitters/langchain_text_splitters
#Existem várias opções de text_splitters

In [26]:
text_splitter = RecursiveCharacterTextSplitter (
    chunk_size = 500, #bom ponto de partida
    chunk_overlap = 75 #10 a 20% do chunk size
)

chunking_documentos = text_splitter.split_documents (documentos)

#Chunks
display (chunking_documentos)

#Os dois primeiros Chunks
####display (chunking_documentos[0].page_content)
####display (chunking_documentos[1].page_content)

#Estatística dos Chunks
####print (f"Número de Documentos: {len(documentos)} \nNúmero de Chunks: {len(chunking_documentos)}")

[Document(metadata={'title': 'Go Forward and Transform.pdf'}, page_content='Go Forward and Transform Arquitetura Transformer e a Evolução dos Modelos de Linguagem Modernos\nEliezer Carvalho eliezer.carvalho.11.8a@gmail.com\nAbstracto'),
 Document(metadata={'title': 'Go Forward and Transform.pdf'}, page_content='Os Transformers [1] constituem uma arquitetura de Redes Neurais Artificiais fundamentada em mecanismos de Attention. Num período recente, estes modelos estabeleceram-se como a base dominante no Processamento de Linguagem Natural, impulsionando o desenvolvimento de Large Language Models (LLMs) com capacidades avançadas de geração e compreensão textual. O presente estudo propõe uma exploração e compreensão da arquitetura dos Transformers. É apresentada uma visão abrangente do ecossistema'),
 Document(metadata={'title': 'Go Forward and Transform.pdf'}, page_content='dos Transformers. É apresentada uma visão abrangente do ecossistema contemporâneo de LLMs, incluindo técnicas e ferra

---------------------------------------------------------------------------

<h1> Embedding </h1>

<p>
Embedding é a etapa onde convertemos os Chunks em representações vetoriais, capturando assim o seu valor semântico. <br>
Nesta etapa é necessário um Embedding Model.
</p>

https://medium.com/@sharanharsoor/the-complete-guide-to-embeddings-and-rag-from-theory-to-production-758a16d747ac

In [27]:
page_content = []
#Para o modelo Embedding entra apenas texto, não metadata

for i in range (len(chunking_documentos)):
    page_content.append(chunking_documentos[i].page_content)

#print (page_content)
#print (page_content[0])
#print (page_content[1])

In [28]:
from transformers import AutoTokenizer, AutoModel
import torch

In [29]:
emb_model = r"C:\Users\Admin\Desktop\models\Bi Encoder"

tokenizer = AutoTokenizer.from_pretrained (emb_model)
modelo = AutoModel.from_pretrained (emb_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2963.20it/s]


In [30]:

#Maneira de fazer Embedding
inputs = tokenizer (page_content, return_tensors = "pt", padding = True, truncation = True) #Padding -> Substitui posições de embeddings não utilizados por [PAD] #Truncation -> Corta quando o número de tokens excede o tamanho máximo do modelo

#print (inputs["input_ids"])
#print (len(inputs["input_ids"]))
#print (tokenizer.decode(inputs["input_ids"][0]))
#print (chunking_documentos[0].page_content)

#Modelo pega em cada token e cria uma representação vetorial chamada de Embedding Dimension (Tamanho depende de cada Embedding Model)
with torch.no_grad():
    outputs = modelo (**inputs) 

#print (outputs)
#print (outputs.keys())
#print (outputs.last_hidden_state)
#print (f"Sequence Length do Primeiro Chunk: {outputs.last_hidden_state[0].shape[0]} \nEmbedding Dimension do Primeiro Chunk: {outputs.last_hidden_state[0].shape[1]}")

#Pooling é o processo de juntar vários vetores (neste caso por token) num único vetor fixo que representa todo o texto.
#Isto é importante porque neste momento temos apenas vetores por cada token do respetivo chunk e nós queremos uma representação final do chunk.
attention_mask = inputs["attention_mask"]
mask = attention_mask.unsqueeze (-1) #Não podemos ter em conta os tokens [PAD]
embeddings = (outputs.last_hidden_state * mask).sum(dim = 1) / mask.sum(dim = 1)

#print (embeddings)
print (f"Texto Chunk 0: {chunking_documentos[0].page_content}\nEmbeddings Chunk 0: {embeddings[0]}")
#print (f"Os {embeddings.shape[0]} Chunks são representados cada, por {embeddings.shape[1]} valores.")


Texto Chunk 0: Go Forward and Transform Arquitetura Transformer e a Evolução dos Modelos de Linguagem Modernos
Eliezer Carvalho eliezer.carvalho.11.8a@gmail.com
Abstracto
Embeddings Chunk 0: tensor([-1.8992e-01,  1.2925e-01, -1.4847e-01, -1.5662e-01, -2.8252e-01,
         3.5681e-02, -4.1027e-02,  1.7028e-01,  6.5550e-02, -1.0021e-01,
         2.4813e-01, -3.4381e-02, -2.5328e-02,  3.4841e-02,  8.4379e-02,
        -1.4614e-01, -1.2742e-01,  1.1262e-01, -2.9663e-02, -2.2915e-01,
         1.1575e-01,  1.6110e-01, -1.1026e-01,  5.4807e-02,  2.9969e-02,
         1.3523e-01,  4.0278e-02, -9.9464e-02,  2.5606e-02, -1.1136e-01,
        -1.5818e-01,  1.2864e-01, -1.3923e-01,  8.8204e-02, -3.0368e-01,
        -5.6732e-02, -2.9568e-02, -7.9869e-02,  2.6827e-02,  1.1953e-01,
        -1.2504e-01, -1.0138e-01,  1.1012e-01, -1.5444e-01,  1.1637e-01,
         6.2560e-02,  5.4829e-02,  4.8120e-02, -3.1983e-02, -1.7352e-01,
         4.3834e-02, -7.4354e-02, -1.2266e-01,  2.1072e-01, -2.2672e-01,
      

---------------------------------------------------------------------------

<h1> Indexing </h1>

<p>
Após os Embeddings é necessário armazenar os mesmos para posterior consulta. <br>
Para tal usamos Bases de Dados Vetoriais.
</p>


In [31]:
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface

C:\Users\Admin\AppData\Local\Temp\ipykernel_10608\2595397039.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores


In [32]:
emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (chunking_documentos, emb_hf) #Este método calcula os Embeddings e mete logo numa base de dados vetorial

#display (db.index.reconstruct(0))
#display (db.docstore._dict)
#display (db.index.ntotal) #Número de Chunks
#display (db.index_to_docstore_id)
#db.save_local("x")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3126.39it/s]


---------------------------------------------------------------------------

https://arxiv.org/pdf/2004.04906

<h1> Retrieval </h1>

<p>
Consiste na capacidade de após receber uma questão (query), conseguir localizar documentos/chunks importantes e relevantes relacionados com a query, na base de dados vetorial.
</p>

<h3> Sparse Retrieval </h3>

<p>
Método clássico. <br>
Procura correspondência direta de palavras, entre a query e os documentos.
</p>

https://huggingface.co/blog/yjoonjang/the-past-and-present-of-sparse-retrieval\

In [33]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

In [34]:
sparse_retrieval = BM25Retriever.from_documents (chunking_documentos, k = 10)

#display (sparse_retrieval)
#display (sparse_retrieval.docs)

<h3> Dense Retrieval </h3>

<p>
Neste tipo de Retrieval ambas as queries e os documentos são transformados em embeddings. <br>
A relevância é calculada usando métricas de similiaridade (produto interno ou similaridade do cosseno) entre os embeddings da query e os embeddings dos documentos.
</p>

https://www.emergentmind.com/topics/dense-retrieval-dr

In [35]:
###Exemplo

#Conversão da query em Embeddings
query = "Em que ano terminou a Segunda Guerra Mundial ?"

embededed_query = emb_hf.embed_query (query)
#display (embededed_query)

#simil = db.similarity_search (query)
#simil_scores = db.similarity_search_with_relevance_scores (query)

#simil_vector = db.similarity_search_by_vector (embededed_query, k = 10)
#simil_vector_scores = db.similarity_search_with_score_by_vector (embededed_query, k = 10)

In [36]:
#Com LangChain é possível converter a Vector Database diretamente para um Dense Retrieval

dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

retrieval = dense_retrieval.invoke (query) #== db.similarity_search_by_vector (embededed_query, k = 10)
#retrieval

<h3> Reciprocal Rank Fusion (RRF) </h3>

<p>
É um método usado para combinar scores. <br>
Muito usado em sistemas Hybrid Retrieval tal como este. <br>
Consiste na criação de um sistema de Ranking entre métodos de Retrieval. 
</p>

$$
RRF (d) = \sum _{i = 1}^{n} \frac {1} {k + rank_i(d)} 
$$

In [37]:
def Reciprocal_Rank_Fusion (rankings, k = 60): #60 é uma variável usada para reduzir o impacto de diferenças pequenas entre ranks
    
    points = {}

    for ranking in rankings:
        for rank, doc in enumerate (ranking):
            id_doc = doc.page_content

            points[id_doc] = points.get(id_doc, 0) + 1 / (k + rank + 1)

    return sorted (points.items(), key = lambda x: x[1], reverse = False) #Dá return dos docs ordenados e respetivas pontuações sorted

In [43]:
###Exemplo

query = "Qual é o mail de Eliezer Carvalho ?"

sparse = sparse_retrieval.invoke (query)
dense = dense_retrieval.invoke (query)

teste = Reciprocal_Rank_Fusion ([sparse, dense])

#display (teste)

text = []
for i in range (len(teste)):
    text.append(teste[i][0])
    #display (teste[i][0])

In [44]:
display (text)

['Como já observámos, durante a fase de pré-treino, o modelo é sujeito a um processo de aprendizagem de padrões estatísticos da linguagem. Posteriormente, são aplicadas técnicas adicionais de adaptação, tais como o Fine-Tuning, incluindo abordagens como o Instruction Fine-Tuning e o Reinforcement Learning from Human Feedback (RLHF). Estas técnicas visam ajustar o comportamento do modelo para uma interação mais alinhada com instruções humanas e preferências de utilização.',
 'em 1958, 1962, 1970, 1994 e 2002. É também o único proprietário permanente da Taça Jules Rimet (posta em jogo em 1930) e ganha em definitivo pelo país que vencesse pela terceira vez o campeonato, o que se deu na competição em 1970, com Pelé, o único jogador tricampeão mundial da história. A Seleção Brasileira é seguida pela Itália (1934, 1938, 1982 e 2006) e Alemanha (1954, 1974, 1990 e 2014) com quatro troféus cada uma. A Argentina é tricampeã, ganhando em (1978, 1986 e 2022), e é a atual',
 '>> \n>> uv pip instal

<h1> Reranking </h1>

Reranking é a fase final do RAG. <br>
No Retrieval encontramos relações entre a Query e os documentos usando a similiridade do cosseno. Isto significa que ambos são analisados de maneira separada usando Bi-Encoders. <br>
Na fase de Reranking, o objetivo é usar um modelo Cross-Encoder que receba como input tanto a Query como a documentação pós retrieval. <br>
Desta maneira, o Cross Encoder consegue encontrar relações mais profundas e complexas entre a Query e os documentos pós Retrieval. <br>

Após o Cross Encoder alimentamos o modelo com top_k de documentos com melhor classificação.

In [45]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder = AutoModelForSequenceClassification.from_pretrained (path)
ce_tokenization = AutoTokenizer.from_pretrained (path)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2320.68it/s]


In [46]:
#### teste

#features = [query] * len (teste), [t[0] for t in teste]

QUERY = [query] * len (teste)
DOCSS = [t[0] for t in teste]
#print (QUERY)
#print (DOCSS)

inputs = ce_tokenization (QUERY, DOCSS, return_tensors = "pt", padding = True, truncation = True)
#print (inputs)

cross_encoder.eval()
with torch.no_grad():
    logits = cross_encoder (**inputs).logits
    #print (logits)

#Organiza os logits com os chunks correspondentes
rerank = sorted(zip(text, logits.tolist()), 
    key = lambda x : x[1],
    reverse = True)

#display (rerank)

In [47]:
feed_llm = [chunk for chunk, scores in rerank [:3]]

print (feed_llm)

['Go Forward and Transform Arquitetura Transformer e a Evolução dos Modelos de Linguagem Modernos\nEliezer Carvalho eliezer.carvalho.11.8a@gmail.com\nAbstracto', 'Cenário 2: O e-mail é spam Y = 1, a IA tem apenas 10% de confiança então Y = 0,1. Perda = −log(0,1) ≈ 2,3 - Cerca de 22 vezes pior! Correto, mas demasiado incerto.\nCenário 3: O e-mail não é spam Y = 0, a IA está 90% confiante de que é, então Y = 0,9. Perda = −log (1−0,9) = −log(0,1) ≈ 2,3 - Penalização igualmente grave por estar confiantemente errado.', 'em 1958, 1962, 1970, 1994 e 2002. É também o único proprietário permanente da Taça Jules Rimet (posta em jogo em 1930) e ganha em definitivo pelo país que vencesse pela terceira vez o campeonato, o que se deu na competição em 1970, com Pelé, o único jogador tricampeão mundial da história. A Seleção Brasileira é seguida pela Itália (1934, 1938, 1982 e 2006) e Alemanha (1954, 1974, 1990 e 2014) com quatro troféus cada uma. A Argentina é tricampeã, ganhando em (1978, 1986 e 202

<h1> LLM ??????</h1>


In [48]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


#model = AutoModelForCausalLM.from_pretrained ("Qwen/Qwen2.5-0.5B-Instruct")
#tokenization = AutoTokenizer.from_pretrained ("Qwen/Qwen2.5-0.5B-Instruct")

#model.save_pretrained (r"C:\Users\Admin\Desktop\models\Qwen2.5-0.5B-Instruct")
#tokenizer.save_pretrained (r"C:\Users\Admin\Desktop\models\Qwen2.5-0.5B-Instruct")



In [49]:
path = r"C:\Users\Admin\Desktop\models\Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer2 = AutoTokenizer.from_pretrained (path)

device

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2915.80it/s]


'cpu'

In [50]:
prompt = "Qual é o mail de Eliezer Carvalho ?"


chat_template = f"""
<|im_start|>
És um assistente especializado em responder exclusivamente com base no contexto fornecido.

Regras:
1. Utiliza apenas a informação da Base de Dados.
2. Não inventes informação.
3. Não uses conhecimento externo.
4. Se a resposta não existir no contexto, responde exatamente:
   "Informação não encontrada na base de dados."
5. Responde de forma breve e direta.
6. Formata a resposta de forma clara.

Base de Dados:
{feed_llm}
<|im_end|>

<|im_start>
{prompt}
<|im_end|>

<|im_start|>
"""

model_inputs = tokenizer2 (chat_template, return_tensors = "pt").to(model.device)



generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

input_length = model_inputs["input_ids"].shape[1]
response_tokens = generated_ids[0][input_length:]

print(tokenizer2.decode(response_tokens, skip_special_tokens = True))

O mail de Eliezer Carvalho é:

"Eliezer.Carvalho.11.8a@gmail.com"

Ele usa esse email para responder às suas perguntas sobre seu trabalho e sua relação com outros indivíduos relevantes.
